In [ ]:
# NOTEBOOK 31 - does the v3 model respond to wind scenarios in a physically sensible way?
# The live system uses the v3 models (ERA5 wind in the inputs and a wind forecast in the decoder). Notebook 12 ran the what-if
# scenarios on the old v2 model only. Here the same three scenarios are run through the 7 v3 models on 300 stored test windows:
#   wind_x2  : wind speed doubled                 (physics: more dispersion, so PM2.5 should fall)
#   wind_nw  : wind turned to blow FROM the north-west at the same speed   (stubble-smoke direction into Delhi)
#   wind_rev : wind direction reversed
# Each is applied to (a) the input wind and the forecast wind together, (b) the input wind only, (c) the forecast wind only,
# to see which path the model actually listens to. Runs on the CPU in a few minutes; nothing is trained or saved except one JSON.
# HOW TO CHECK: this cell only prints a note.
print('v3 what-if check on 300 windows')

In [ ]:
import os, json, time, copy, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
OUT    = os.path.join(V2_DIR, 'ecotwin_outputs')
SCALER_DIR, PLOT_DIR   = os.path.join(OUT, 'scalers'), os.path.join(OUT, 'plots')
MODEL_DIR, RESULTS_DIR = os.path.join(OUT, 'models'),  os.path.join(OUT, 'results')

DATASET_CSV     = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
STATIONS_EXCEL  = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')
ERA5_WIND_NPY   = os.path.join(V2_DIR, 'openmeteo_reanalysis_uv.npy')
DYN_ADJ_OM_NPY  = os.path.join(V2_DIR, 'dynamic_adj_om.npy')
STATIC_ADJ_NPY  = os.path.join(V2_DIR, 'static_adj_matrix.npy')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_WINDOW, HORIZON = 48, 24
BATCH_SIZE = 16
LOSS_VOL_ALPHA = 1.0
SEEDS = [42, 7, 123, 2024, 999]

cfg = json.load(open(os.path.join(MODEL_DIR, 'v3_config.json')))
FEATURE_COLS = cfg['feature_cols']; F = cfg['in_features']
HIDDEN_DIM, GRU_LAYERS = cfg['hidden_dim'], cfg['gru_layers']
ATTN_HEADS, DROPOUT, GCN_HOPS = cfg['attn_heads'], cfg['dropout'], cfg['gcn_hops']
N_FUT = cfg['n_fut']
print(f'Device {DEVICE} | F={F} | n_fut={N_FUT} | seeds {SEEDS}')

In [ ]:
df = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()
df = df.sort_values(['Station', 'Timestamp']).reset_index(drop=True)
df_st = pd.read_excel(STATIONS_EXCEL)
df_st.columns = df_st.columns.str.strip().str.lower()
df_st['station'] = df_st['station'].str.strip()
station_names = df_st['station'].tolist()
N = len(station_names)

timestamps = np.array(sorted(df['Timestamp'].unique())); T = len(timestamps)
era5 = np.load(ERA5_WIND_NPY).astype(np.float32)
era5_spd = np.hypot(era5[..., 0], era5[..., 1])
dyn_adj = np.load(DYN_ADJ_OM_NPY).astype(np.float32)                 # A(t) from ERA5 wind (T,N,N)
static_adj = np.load(STATIC_ADJ_NPY).astype(np.float32)
np.fill_diagonal(static_adj, 0.0)
assert dyn_adj.shape == (T, N, N)
print(f'T={T}  dyn_adj(ERA5) {dyn_adj.shape}  static edges {int((static_adj>0).sum())}')

In [ ]:
BASE_FEATS = ['pm2.5','pm10','no2','so2','co','o3','rh','at']
g = df.groupby('Station', sort=False)
df['pm_roll6_mean']  = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).mean())
df['pm_roll6_std']   = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).std()).fillna(0.0)
df['pm_roll24_mean'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).mean())
df['pm_roll24_std']  = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).std()).fillna(0.0)
df['pm_delta3']      = g['pm2.5'].transform(lambda s: s - s.shift(3)).fillna(0.0)
tsg = df.groupby('Timestamp', sort=False)['pm2.5']
df['city_mean_pm']  = tsg.transform('mean')
df['city_max_pm']   = tsg.transform('max')
df['pm_minus_city'] = df['pm2.5'] - df['city_mean_pm']
hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month
df['hour_sin'] = np.sin(2*np.pi*hh/24);      df['hour_cos'] = np.cos(2*np.pi*hh/24)
df['dow_sin']  = np.sin(2*np.pi*dow/7);      df['dow_cos']  = np.cos(2*np.pi*dow/7)
df['mon_sin']  = np.sin(2*np.pi*(mon-1)/12); df['mon_cos']  = np.cos(2*np.pi*(mon-1)/12)
df['is_weekend'] = (dow >= 5).astype(float)

CPCB_COLS = (BASE_FEATS +
    ['pm_roll6_mean','pm_roll6_std','pm_roll24_mean','pm_roll24_std','pm_delta3',
     'city_mean_pm','city_max_pm','pm_minus_city',
     'hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend'])
Xc = np.zeros((T, N, len(CPCB_COLS)), dtype=np.float32)
for fi, col in enumerate(CPCB_COLS):
    wide = df.pivot_table(index='Timestamp', columns='Station', values=col).reindex(columns=station_names)
    Xc[:, :, fi] = np.nan_to_num(wide.values.astype(np.float32))

built = (BASE_FEATS + ['u_wind', 'v_wind', 'wspd'] + CPCB_COLS[len(BASE_FEATS):])
assert built == FEATURE_COLS, 'feature order mismatch vs v3_config'
X_raw = np.concatenate([
    Xc[:, :, :len(BASE_FEATS)],
    era5[:, :, 0:1], era5[:, :, 1:2], era5_spd[:, :, None],
    Xc[:, :, len(BASE_FEATS):],
], axis=2).astype(np.float32)
Y_raw = X_raw[:, :, 0].copy()

TIME_COLS = ['hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend']
TIME_IDX  = [FEATURE_COLS.index(c) for c in TIME_COLS]
U_IDX, V_IDX, W_IDX = FEATURE_COLS.index('u_wind'), FEATURE_COLS.index('v_wind'), FEATURE_COLS.index('wspd')
FUT_IDX   = TIME_IDX + [U_IDX, V_IDX, W_IDX]
print(f'F={X_raw.shape[2]}  NaNs {int(np.isnan(X_raw).sum())}')

In [ ]:
# CPU, geometry and the graph recipe (checked against the stored A(t) built from ERA5 wind)
import pickle
DEVICE = torch.device('cpu')
lats = df_st['latitude'].values.astype(float); lons = df_st['longitude'].values.astype(float)
THRESHOLD_KM = 12.0
dist = np.zeros((N, N)); du = np.zeros((N, N)); dv = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i != j:
            p1, p2 = np.radians(lats[i]), np.radians(lats[j])
            a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lons[j] - lons[i]) / 2) ** 2
            dist[i, j] = 12742 * np.arcsin(np.sqrt(a))
            de = (lons[j] - lons[i]) * np.cos(np.radians(lats[i])); dn = lats[j] - lats[i]; nr = np.hypot(de, dn)
            if nr > 0: du[i, j], dv[i, j] = de / nr, dn / nr
within = (dist > 0) & (dist <= THRESHOLD_KM)
def compute_At_vec(u, v, eps=1e-6):
    align = u[:, None] * du + v[:, None] * dv
    A = np.where(within & (align > 0.0), align / (dist + eps), 0.0).astype(np.float32)
    rs = A.sum(1, keepdims=True); rs[rs == 0] = 1.0
    return A / rs
chk = np.abs(np.stack([compute_At_vec(era5[t, :, 0], era5[t, :, 1]) for t in (100, 5000, 12000)]) - dyn_adj[[100, 5000, 12000]]).max()
print('graph recipe vs stored A(t), max abs difference:', float(chk))

# HOW TO CHECK: the difference is about 1e-6 or smaller. If it is large, stop and tell me.

In [ ]:
class WATGCNv3Ablate(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads, forecast_horizon,
                 n_stations, dropout=0.2, n_fut=10, gcn_hops=2, graph_mode='dynamic', static_adj=None):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin  = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True, dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.fut_enc = nn.Sequential(nn.Linear(n_fut, H), nn.ReLU(), nn.Linear(H, H))
        self.decoder = nn.Sequential(nn.Linear(H*3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH, self.graph_mode = H, forecast_horizon, graph_mode
        sa = torch.eye(n_stations) if static_adj is None else torch.from_numpy(static_adj.astype('float32'))
        self.register_buffer('static_adj', sa, persistent=False)

    @staticmethod
    def _n(A):
        return A / A.sum(-1, keepdim=True).clamp(min=1e-6)

    def forward(self, X, At, Afc, Fut, Anchor):
        B, Tin, n, _ = X.shape; dev = X.device; I = torch.eye(n, device=dev)
        h = self.act(self.input_proj(X))
        if self.graph_mode == 'none':
            for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
                h = self.act(nrm(lin(h))) + h
        else:
            if self.graph_mode == 'dynamic':
                A = self._n(At + I.view(1, 1, n, n))
            else:
                A = self._n((self.static_adj + I).view(1, 1, n, n)).expand(B, Tin, n, n)
            for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
                h = self.act(nrm(lin(torch.matmul(A, h)))) + h
        hf = h.permute(0, 2, 1, 3).reshape(B*n, Tin, self.H)
        gg, _ = self.gru(hf); gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        if self.graph_mode == 'none':
            ctx_fc = ctx_rep
        elif self.graph_mode == 'dynamic':
            ctx_fc = torch.matmul(self._n(Afc + I.view(1, 1, n, n)), ctx_rep)
        else:
            ctx_fc = torch.matmul(self._n((self.static_adj + I).view(1, 1, n, n)), ctx_rep)
        fut_emb = self.fut_enc(Fut)
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, fut_emb], dim=-1)).squeeze(-1)
        return resid + Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)

In [ ]:
# scalers, the 7 v3 models and the sample of test windows
feat_scaler = pickle.load(open(os.path.join(SCALER_DIR, 'v3_feature_scaler.pkl'), 'rb'))
log_scaler  = pickle.load(open(os.path.join(SCALER_DIR, 'v3_log_scaler.pkl'), 'rb'))
def from_log(a): return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))
def to_log(a):   return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)
Yl = to_log(Y_raw)
SEEDS7 = cfg['seeds']; STATIC = static_adj.copy()
MODELS = []
for sd in SEEDS7:
    m = WATGCNv3Ablate(F, HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, N, DROPOUT, N_FUT, GCN_HOPS, graph_mode='dynamic', static_adj=STATIC).to(DEVICE)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'watgcn_v3_seed{sd}.pt'), map_location=DEVICE)); m.eval(); MODELS.append(m)
val_end = int(T * 0.85)
test_starts = np.arange(val_end, T - (INPUT_WINDOW + HORIZON) + 1)
SAMPLE = test_starts[np.linspace(0, len(test_starts) - 1, 300).astype(int)]
fut_spd = np.array([era5_spd[g + INPUT_WINDOW: g + INPUT_WINDOW + HORIZON].mean() for g in SAMPLE])
w33, w66 = np.percentile(fut_spd, [33, 66])
REG = {'calm': fut_spd <= w33, 'breezy': (fut_spd > w33) & (fut_spd <= w66), 'windy': fut_spd > w66}
print(f'{len(MODELS)} models | {len(SAMPLE)} windows | forecast-period wind speed terciles {w33:.2f} / {w66:.2f} m/s')

In [ ]:
# run the scenarios
def perturb(u, v, kind):
    if kind == 'wind_x2': return u * 2.0, v * 2.0
    if kind == 'wind_nw':
        s = np.hypot(u, v); return s * 0.7071, -s * 0.7071       # wind FROM the north-west: air moves south-east
    if kind == 'wind_rev': return -u, -v
    return u, v
@torch.no_grad()
def run(kind, where):          # where: 'both' | 'input' | 'future'
    out = np.zeros((len(SAMPLE), HORIZON, N), dtype=np.float32)
    for b0 in range(0, len(SAMPLE), 30):
        gg = SAMPLE[b0:b0 + 30]; Xb, Ab, Afb, Fb, Anb = [], [], [], [], []
        for g in gg:
            x = X_raw[g:g + INPUT_WINDOW].copy(); u, v = x[:, :, U_IDX].copy(), x[:, :, V_IDX].copy()
            if where in ('both', 'input'): u, v = perturb(u, v, kind)
            x[:, :, U_IDX], x[:, :, V_IDX], x[:, :, W_IDX] = u, v, np.hypot(u, v)
            Xb.append(feat_scaler.transform(x.reshape(-1, F)).reshape(INPUT_WINDOW, N, F).astype(np.float32))
            Ab.append(np.stack([compute_At_vec(u[h], v[h]) for h in range(INPUT_WINDOW)]))
            f = X_raw[g + INPUT_WINDOW: g + INPUT_WINDOW + HORIZON].copy(); uf, vf = f[:, :, U_IDX].copy(), f[:, :, V_IDX].copy()
            if where in ('both', 'future'): uf, vf = perturb(uf, vf, kind)
            f[:, :, U_IDX], f[:, :, V_IDX], f[:, :, W_IDX] = uf, vf, np.hypot(uf, vf)
            Fb.append(feat_scaler.transform(f.reshape(-1, F)).reshape(HORIZON, N, F)[:, :, FUT_IDX].astype(np.float32))
            Afb.append(np.stack([compute_At_vec(uf[h], vf[h]) for h in range(HORIZON)]))
            Anb.append(Yl[g + INPUT_WINDOW - 1])
        T_ = lambda a: torch.from_numpy(np.stack(a)).to(DEVICE)
        outs = np.stack([m(T_(Xb), T_(Ab), T_(Afb), T_(Fb), T_(Anb)).cpu().numpy() for m in MODELS])
        out[b0:b0 + len(gg)] = from_log(outs.mean(0))
    return out
t0 = time.time()
base = run('baseline', 'both')
print(f'baseline done in {time.time()-t0:.0f}s; mean forecast {base.mean():.1f} ug/m3')
res = {}
for kind in ('wind_x2', 'wind_nw', 'wind_rev'):
    for where in ('both', 'input', 'future'):
        res[(kind, where)] = run(kind, where) - base
    print(f'{kind} done ({time.time()-t0:.0f}s)')

In [ ]:
# results: mean change in the 24-hour mean forecast (ug/m3), share of windows where PM2.5 goes UP, and by wind regime
print(f'mean forecast {base.mean():.1f} ug/m3 | change = scenario minus baseline | negative = cleaner air')
print()
print(f"{'scenario':10}{'applied to':>11}{'mean change':>13}{'% windows up':>14}   calm  breezy  windy")
summary = {}
for (kind, where), d in res.items():
    wmean = d.mean(axis=(1, 2)); regs = {k: float(wmean[m_].mean()) for k, m_ in REG.items()}
    summary[f'{kind}|{where}'] = {'mean_change': float(wmean.mean()), 'pct_windows_up': float(100 * (wmean > 0).mean()), 'by_regime': regs}
    print(f"{kind:10}{where:>11}{wmean.mean():>+13.1f}{100 * (wmean > 0).mean():>13.0f}%  " + ''.join(f'{regs[k]:>+7.1f}' for k in ('calm', 'breezy', 'windy')))
print()
print('wind_x2 (both) change by forecast hour:', {f'+{h+1}h': round(float(res[('wind_x2', 'both')][:, h].mean()), 1) for h in (0, 2, 5, 11, 17, 23)})
print()
x2 = summary['wind_x2|both']
print('VERDICT: doubling the wind', 'LOWERS' if x2['mean_change'] < -1 else 'does NOT clearly lower', f"PM2.5 on average ({x2['mean_change']:+.1f} ug/m3, "
      f"{x2['pct_windows_up']:.0f}% of windows go up). Physics expects a fall, mostly in calm and breezy windows.")
json.dump({'baseline_mean': float(base.mean()), 'windows': int(len(SAMPLE)), 'summary': summary}, open(os.path.join(RESULTS_DIR, 'v3_whatif_check.json'), 'w'), indent=2)

# HOW TO READ: the model should show a fall for doubled wind in calm and breezy windows. If it is flat or positive, the model's wind response is not
# reliable enough to present the what-if as a physical forecast.